# LO3 — LangChain 도구 호출 (Tool Calling)

## 학습 목표

- `@tool` 데코레이터로 일반 파이썬 함수를 LangChain 도구로 변환할 수 있다.
- `bind_tools`로 모델에 도구 목록을 연결할 수 있다.
- `AIMessage.tool_calls`를 읽어 도구를 실행하고 `ToolMessage`로 결과를 되돌리는 수동 루프를 직접 구현할 수 있다.

## 실습 흐름

도구 호출은 네 단계를 거칩니다. 사용자가 질문하면 모델이 어떤 도구를 어떤 인자로 부를지 결정해 호출 요청을 반환하고, 코드가 그 요청대로 실제 함수를 실행한 뒤, 결과를 모델에 되돌리면 모델이 최종 답변을 완성합니다. 이 한 바퀴를 직접 코드로 도는 과정을 수동 루프라고 부릅니다.

> 실제 실행에는 API 키가 필요합니다. 강의 직전 모델명과 가격을 재확인하십시오.

## 1단계: 설치 (핀 고정)

강의 시점 검증 버전으로 핀을 고정합니다 (2026-06-04 기준).

In [ ]:
!pip install -U "langchain==1.3.4" "langgraph==1.2.4" langchain-openai langchain-google-genai

## 2단계: API 키 설정

Colab에서는 좌측 열쇠 메뉴에 `OPENAI_API_KEY`를 한 번 등록하면 이후 모든 LO에서 재사용됩니다. 로컬에서 실행한다면 아래 주석을 참고해 환경변수나 `.env` 파일로 키를 주입하십시오.

In [ ]:
import os
from google.colab import userdata

# Colab 좌측 열쇠 메뉴에 OPENAI_API_KEY를 등록한 뒤 불러옵니다
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 로컬/IDE 대안:
#   os.environ["OPENAI_API_KEY"] = "sk-..."
# 또는 프로젝트 루트 .env 파일에 OPENAI_API_KEY=sk-... 작성 후
#   from dotenv import load_dotenv; load_dotenv()

## 3단계: 모델과 도구 정의

`@tool` 데코레이터를 붙이면 함수 이름이 도구 이름이 되고, 타입 힌트가 인자 스키마가 되며, docstring이 도구 설명으로 쓰입니다.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# 검증된 실습 모델 (대안: google-genai:gemini-2.5-flash)
# 강의 직전 최신 모델과 가격을 재확인하십시오.
model = init_chat_model("openai:gpt-5.4-mini")


@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""  # docstring이 도구 설명으로 사용됨
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱한다."""
    return a * b


# 도구 메타데이터 확인: 이름, 설명, 인자 스키마가 자동 추출됨
print(add.name, "|", add.description, "|", add.args)
# 예상 출력:
# add | 두 정수를 더한다. | {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}

### 체크포인트

- `add.description`이 docstring 그대로 출력되는지 확인하십시오.
- `add.args`에 타입 힌트로부터 추출된 인자 스키마가 담겨 있는지 확인하십시오.

## 4단계: 모델에 도구 바인딩

`model.bind_tools([도구들])`로 모델에 도구 목록을 묶어 줍니다. 이름으로 도구를 다시 찾기 위한 사전도 함께 만듭니다.

In [ ]:
tools = [add, multiply]
tool_map = {t.name: t for t in tools}  # 이름으로 도구를 찾기 위한 사전
model_with_tools = model.bind_tools(tools)

## 5단계: 수동 루프 (tool_calls 읽기, ToolMessage 되돌리기, 재호출)

`bind_tools`로 묶은 모델은 답 대신 `AIMessage.tool_calls`에 호출 요청 목록을 담아 돌려줍니다. 각 요청은 `{name, args, id}` 형태입니다. 코드가 실제 도구를 실행한 뒤 결과를 `ToolMessage`로 감싸 되돌리고, 호출 요청이 남아 있는 동안 반복합니다.

In [ ]:
from langchain.messages import HumanMessage, ToolMessage  # v1 경로 (langchain_core.messages도 동작)

# 1단계: 사용자 질문 (뒤 계산이 앞 결과에 의존하는 다단계 질문)
messages = [HumanMessage("3 더하기 5를 한 다음 그 결과에 4를 곱하면?")]

# 2단계: 모델이 답 대신 호출 요청(tool_calls)을 반환
ai = model_with_tools.invoke(messages)
messages.append(ai)
print("첫 호출 요청:", ai.tool_calls)  # [{'name','args','id'}, ...]
# 예상 출력(의존 관계가 있어 먼저 덧셈만 요청):
# 첫 호출 요청: [{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_abc'}]

# 3~4단계 반복: 호출 요청이 남아 있는 동안 실행하고 결과를 되돌린 뒤 재호출
#   덧셈 결과(8)를 받은 모델이 곱셈을 요청하므로 tool_calls가 또 채워질 수 있습니다.
#   tool_calls가 빌 때까지 반복해야 최종 답에 도달합니다.
while ai.tool_calls:
    for call in ai.tool_calls:
        chosen = tool_map[call["name"]]  # 요청한 이름의 도구 선택
        result = chosen.invoke(call["args"])  # 실제 함수 실행
        # tool_call_id를 요청 id와 똑같이 맞춰야 모델이 결과를 짝지을 수 있음
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    ai = model_with_tools.invoke(messages)
    messages.append(ai)

print("최종 답변:", ai.content)
# 예상 출력: 최종 답변: 3 더하기 5는 8이고, 거기에 4를 곱하면 32입니다.

### 체크포인트

- `bind_tools`로 묶은 모델의 첫 응답에 `content`가 비고 `tool_calls`가 채워지는지 확인하십시오.
- `ToolMessage`의 `tool_call_id`를 요청 `id`와 맞추지 않으면 모델이 결과를 짝짓지 못합니다.
- 복합 질문에서 `add`와 `multiply`가 의존 순서대로 호출되는지 관찰하십시오.

**변형 포인트**: 질문을 "서울과 부산의 현재 기온을 더하면?"처럼 서로 독립적인 두 조회로 바꾸면, 모델은 첫 응답에서 두 호출을 `tool_calls`에 한꺼번에 담아 돌려줍니다(병렬 호출). 반대로 위 예처럼 뒤 계산이 앞 결과에 기대면 호출이 단계별로 나뉩니다. `tool_calls`의 길이를 출력해 직접 비교해 보십시오.

---

> **여기까지가 핵심입니다. 아래는 선택(심화)입니다.**

## 6단계: tool_choice로 도구 사용 강제 (선택 실습)

기본값 `auto`에서는 모델이 도구를 쓸지 스스로 정하지만, `any`를 주면 반드시 하나는 부릅니다.

In [ ]:
# 기본값(auto)에서는 인사에 도구를 안 부르지만, any를 주면 반드시 하나는 부릅니다.
forced = model.bind_tools(tools, tool_choice="any")  # 도구 중 아무거나 반드시 사용
ai_forced = forced.invoke([HumanMessage("아무 인사나 해줘")])
print("강제 호출:", ai_forced.tool_calls)  # 인사뿐이어도 도구를 부르려 시도함
# add처럼 특정 도구 이름을 주면 그 도구만, none을 주면 도구를 전혀 못 부릅니다.

### 체크포인트

- `tool_choice="any"`를 주면 도구가 필요 없는 인사에도 모델이 도구를 부르려 하는지 확인하십시오.
- 모델이 도구를 부르지 않으면 `tool_calls`가 빈 목록이며, 이때는 루프를 건너뛰고 `ai.content`를 그대로 씁니다.
- 도구가 한 번에 여러 개 호출될 수 있으므로 반드시 목록을 순회해 처리합니다.

이 수동 루프(반복 처리, 병렬 호출, 종료 조건)를 LangGraph가 자동으로 돌려 주는 방식은 이후 LangGraph 워크플로우와 Agent 구현 LO에서 다룹니다.